# Phase 1: baseline pass^k curve (stock agent)

**Goal:** the full pass^k curve for the stock τ-bench agent. This is the baseline every KiL arm is compared with.

**Run:** all 50 airline tasks × 8 trials = 400 dialogues. pass^k for k = 1..8, with a 95% bootstrap CI over tasks.

**Settings:** all frozen in `settings.py`, the same as the 50-task run in 03. 06–08 import the same file, so every arm runs under the same conditions.
- Model: `Qwen/Qwen3-32B-AWQ` on vLLM, A100 40GB, max length 32768
- Agent and user simulator: the same model on the same server
- Agent thinking on, user thinking off
- Agent reply ≤ 8192 tokens, call timeout 1800 s
- Empty agent reply = normal fail
- Agent temperature 0.0, concurrency 4 (vLLM batching can change results, so it stays fixed)
- Policy: prose `wiki.md`, as τ-bench ships it

**pass^k:** per task, C(passes, k) / C(8, k), then the mean over all 50 tasks. Same formula as τ-bench's own code.

**Time:** about 0.7 min per task × 400 ≈ 4.5–4.7 hours.

**Safe to stop:** each trial has its own folder, and τ-bench saves after every task. After a disconnect, run the cells again: finished tasks are skipped, a file cut off mid-write is ignored (its tasks run again), and tasks that ended in an API error run again. A trial counts as complete only with all 50 tasks.

**How to run:** cells one by one, top to bottom. Run the server cell only once per session.

In [ ]:
# mount Google Drive so results are saved
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%writefile /content/drive/MyDrive/cmsc691-10-project2/settings.py
# frozen settings for every arm (baseline, validator, shallow, semi-deep, deep)
# taken from the 50-task run in 03. Change one, and every arm must be rerun.
MODEL = "Qwen/Qwen3-32B-AWQ"     # 4-bit, on vLLM
SERVED = "qwen3-32b"             # short name tau-bench uses, agent and user
MAX_LEN = 32768                  # vLLM max model length
GPU_MEM = "0.90"                 # vLLM gpu-memory-utilization
AGENT_THINKING = True            # agent thinks, user simulator does not
AGENT_MAX_TOKENS = 8192          # cap on one agent reply
CALL_TIMEOUT = 1800              # seconds per model call
EMPTY_REPLY_IS_FAIL = True       # empty agent reply -> "" -> normal fail
TEMPERATURE = 0.0                # agent temperature
CONCURRENCY = 4                  # same as 03; batching changes results, so keep it fixed
ENV = "airline"
N_TASKS = 50
N_TRIALS = 8

In [ ]:
# check the GPU and load the frozen settings
import subprocess, sys, importlib
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
print(out)
assert int(out.split(", ")[1]) >= 40000, "Need an A100. Runtime → Change runtime type → A100."

sys.path.insert(0, "/content/drive/MyDrive/cmsc691-10-project2")
importlib.invalidate_caches()
import settings; importlib.reload(settings)
from settings import *
print({k: v for k, v in vars(settings).items() if k.isupper()})

In [ ]:
# go into the repo and install everything
# Colab drops packages on restart, so I run this every time
%cd /content/drive/MyDrive/cmsc691-10-project2
!pip install -q -e .
!pip install -q mcp
!pip install -q vllm
print("installed")

In [ ]:
# start the vLLM server in the background
# run this cell ONCE per session; each run starts another server
import subprocess
cmd = ["vllm", "serve", MODEL,
       "--served-model-name", SERVED,
       "--max-model-len", str(MAX_LEN),
       "--gpu-memory-utilization", GPU_MEM,
       "--enable-auto-tool-choice", "--tool-call-parser", "hermes",
       "--reasoning-parser", "qwen3",
       "--port", "8000"]
server = subprocess.Popen(cmd, stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)
print("server starting, pid", server.pid)

In [ ]:
# wait until the server is ready (up to 20 min)
import time, requests
for i in range(120):
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=5).ok:
            print(f"ready after {i*10}s")
            break
    except Exception:
        pass
    if server.poll() is not None:
        print("server died, real error:")
        break
    time.sleep(10)
else:
    print("not ready after 20 min")
!grep -iE "error|out of memory" /content/vllm.log | tail -n 5

In [ ]:
# smoke test: can Qwen call a tool?
import json
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="none")
tools = [{"type": "function", "function": {
    "name": "get_reservation", "description": "Look up a reservation",
    "parameters": {"type": "object", "properties": {"reservation_id": {"type": "string"}},
                   "required": ["reservation_id"]}}}]
r = client.chat.completions.create(
    model=SERVED, messages=[{"role": "user", "content": "Look up reservation ABC123."}],
    tools=tools, extra_body={"chat_template_kwargs": {"enable_thinking": False}})
msg = r.choices[0].message
print("✅ tool call works" if msg.tool_calls else f"❌ no tool call: {msg.content}")

In [ ]:
%%writefile /content/drive/MyDrive/cmsc691-10-project2/pilot_vllm.py
# run tau-bench against my local vLLM server
# every setting comes from settings.py, so all arms share them
import sys
import litellm
from settings import AGENT_THINKING, AGENT_MAX_TOKENS, CALL_TIMEOUT, EMPTY_REPLY_IS_FAIL

# hide litellm's "Give Feedback" lines, they flood the output
litellm.suppress_debug_info = True

_real = litellm.completion

def set_thinking(*args, **kwargs):
    # the agent's calls carry tools, the user simulator's calls don't
    is_agent = bool(kwargs.get("tools"))
    think = AGENT_THINKING if is_agent else False
    # copy first, so I never change tau-bench's own dict
    extra = dict(kwargs.get("extra_body") or {})
    extra["chat_template_kwargs"] = {"enable_thinking": think}
    kwargs["extra_body"] = extra
    if is_agent:
        kwargs.setdefault("max_tokens", AGENT_MAX_TOKENS)
    kwargs["timeout"] = CALL_TIMEOUT
    res = _real(*args, **kwargs)
    # an empty reply (thinking only) crashes tau-bench's scoring
    # turn it into "" so it counts as a normal fail
    msg = res.choices[0].message
    if EMPTY_REPLY_IS_FAIL and msg.content is None and not msg.tool_calls:
        msg.content = ""
    return res

# swap it in BEFORE tau-bench imports completion
litellm.completion = set_thinking

sys.argv[0] = "run.py"
from run import main  # run.py is in the repo folder
main()

In [ ]:
%%writefile /content/drive/MyDrive/cmsc691-10-project2/passk.py
# pass^k for tau-bench runs that are saved one trial per folder
#   results_dir/trial_1/*.json, results_dir/trial_2/*.json, ...
# A task's row with an API error counts as a fail (same as tau-bench),
# unless a later rerun of that task in the same trial succeeded without error.
import glob
import json
import os
import random
from math import comb


def is_error(row):
    return "error" in (row.get("info") or {})


def load_trial(folder):
    """Best row per task in one trial folder: a row without error beats a row with error."""
    best = {}
    for f in sorted(glob.glob(os.path.join(folder, "*.json")), key=os.path.getmtime):
        try:
            rows = json.load(open(f))
        except json.JSONDecodeError:      # file cut off mid-write by a disconnect
            continue
        for r in rows:
            t = r["task_id"]
            if t not in best or (is_error(best[t]) and not is_error(r)):
                best[t] = r
    return best


def done_tasks(folder, retry_errors=True):
    """Task ids already finished in this trial."""
    best = load_trial(folder)
    return {t for t, r in best.items() if not (retry_errors and is_error(r))}


def table(results_dir, n_tasks=50):
    """{trial: {task_id: row}} for every trial folder found."""
    out = {}
    for d in sorted(glob.glob(os.path.join(results_dir, "trial_*"))):
        out[int(d.rsplit("_", 1)[1])] = load_trial(d)
    return out


def success_counts(tab, tasks):
    """c[task] = number of trials where the task passed."""
    return {t: sum(1 for tr in tab.values() if t in tr and tr[t]["reward"] >= 1 - 1e-6) for t in tasks}


def pass_hat_k(counts, n, k):
    """tau-bench's unbiased pass^k: mean over tasks of C(c, k) / C(n, k)."""
    return sum(comb(c, k) / comb(n, k) for c in counts.values()) / len(counts)


def curve(results_dir, n_tasks=50, boot=2000, seed=0):
    """pass^k for k = 1..n with a 95% bootstrap CI over tasks.
    Uses only trials that have all n_tasks tasks."""
    tab = table(results_dir, n_tasks)
    full = {k: v for k, v in tab.items() if len(v) == n_tasks}
    n = len(full)
    if n == 0:
        return n, []
    tasks = list(range(n_tasks))
    counts = success_counts(full, tasks)
    rng = random.Random(seed)
    rows = []
    for k in range(1, n + 1):
        est = pass_hat_k(counts, n, k)
        samples = []
        for _ in range(boot):
            pick = [counts[rng.choice(tasks)] for _ in tasks]
            samples.append(sum(comb(c, k) / comb(n, k) for c in pick) / len(pick))
        samples.sort()
        rows.append({"k": k, "pass^k": est,
                     "ci_low": samples[int(0.025 * boot)], "ci_high": samples[int(0.975 * boot) - 1]})
    return n, rows


def errors(results_dir):
    """(trial, task_id, error text) for every task whose best row is still an API error."""
    out = []
    for trial, rows in table(results_dir).items():
        for t, r in sorted(rows.items()):
            if is_error(r):
                out.append((trial, t, r["info"]["error"][:150]))
    return out

In [ ]:
# run the baseline: 8 trials, one folder each, skip what is done
import os, sys, time, json, importlib
os.environ["HOSTED_VLLM_API_BASE"] = "http://localhost:8000/v1"
os.environ["HOSTED_VLLM_API_KEY"] = "none"
sys.path.insert(0, "/content/drive/MyDrive/cmsc691-10-project2")
importlib.invalidate_caches()
import passk; importlib.reload(passk)

%cd /content/drive/MyDrive/cmsc691-10-project2
OUT = "results_baseline"
os.makedirs("/content/logs", exist_ok=True)
# record the exact settings next to the results
json.dump({**{k: v for k, v in vars(settings).items() if k.isupper()}, "policy": "prose wiki.md"},
          open(f"{OUT}_config.json", "w"), indent=2)

start_all = time.time()
for trial in range(1, N_TRIALS + 1):
    folder = f"{OUT}/trial_{trial}"
    os.makedirs(folder, exist_ok=True)
    todo = sorted(set(range(N_TASKS)) - passk.done_tasks(folder))
    if not todo:
        print(f"trial {trial}: done, skip")
        continue
    print(f"trial {trial}: running {len(todo)} tasks ...", end=" ", flush=True)
    t0 = time.time()
    ids = " ".join(map(str, todo))
    # full log goes to /content/logs, so the notebook stays short
    !python pilot_vllm.py --env {ENV} --agent-strategy tool-calling \
      --model {SERVED} --model-provider hosted_vllm \
      --user-model {SERVED} --user-model-provider hosted_vllm \
      --user-strategy llm --num-trials 1 --temperature {TEMPERATURE} \
      --task-ids {ids} --max-concurrency {CONCURRENCY} \
      --log-dir {folder} > /content/logs/trial_{trial}.log 2>&1
    left = N_TASKS - len(passk.done_tasks(folder))
    print(f"{(time.time() - t0) / 60:.1f} min, {left} not done")
print(f"\nall trials: {(time.time() - start_all) / 60:.1f} min this session")

In [ ]:
# progress: tasks done and API errors per trial
import importlib, passk; importlib.reload(passk)
tab = passk.table("/content/drive/MyDrive/cmsc691-10-project2/results_baseline")
for trial in range(1, N_TRIALS + 1):
    rows = tab.get(trial, {})
    errs = sum(passk.is_error(r) for r in rows.values())
    passed = sum(r["reward"] >= 1 - 1e-6 for r in rows.values())
    print(f"trial {trial}: {len(rows):2}/{N_TASKS} tasks | passed {passed:2} | API errors {errs}")
for trial, task, err in passk.errors("/content/drive/MyDrive/cmsc691-10-project2/results_baseline"):
    print(f"  error trial {trial} task {task}: {err}")

In [ ]:
# pass^k curve with 95% CI, saved to Drive
import os, csv
import matplotlib.pyplot as plt
import importlib, passk; importlib.reload(passk)

n, rows = passk.curve("/content/drive/MyDrive/cmsc691-10-project2/results_baseline", N_TASKS)
print(f"complete trials used: {n}")
print(f"{'k':>2}  {'pass^k':>6}  95% CI")
for r in rows:
    print(f"{r['k']:>2}  {r['pass^k']:6.3f}  [{r['ci_low']:.3f}, {r['ci_high']:.3f}]")

if rows:
    with open("/content/drive/MyDrive/cmsc691-10-project2/results_baseline_passk.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)
    ks = [r["k"] for r in rows]
    plt.figure(figsize=(6, 4))
    plt.fill_between(ks, [r["ci_low"] for r in rows], [r["ci_high"] for r in rows], alpha=0.2, color="#1A1A1A")
    plt.plot(ks, [r["pass^k"] for r in rows], "o-", color="#1A1A1A", label="Stock agent (prose policy)")
    plt.xlabel("k (trials that must all pass)"); plt.ylabel("pass^k")
    plt.title("Baseline pass^k, τ-bench airline, Qwen3-32B")
    plt.ylim(0, max(0.5, rows[0]["ci_high"] + 0.05)); plt.xticks(ks); plt.grid(alpha=0.3); plt.legend()
    os.makedirs("/content/drive/MyDrive/cmsc691-10-project2/figures", exist_ok=True)
    plt.savefig("/content/drive/MyDrive/cmsc691-10-project2/figures/baseline_passk.png", dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
# which tasks are consistent? passes out of 8, per task
import importlib, passk; importlib.reload(passk)
tab = passk.table("/content/drive/MyDrive/cmsc691-10-project2/results_baseline")
full = {t: v for t, v in tab.items() if len(v) == N_TASKS}
c = passk.success_counts(full, range(N_TASKS))
always = [t for t, x in c.items() if x == len(full)]
never = [t for t, x in c.items() if x == 0]
some = [t for t, x in c.items() if 0 < x < len(full)]
print(f"trials: {len(full)}")
print(f"always pass ({len(always)}): {always}")
print(f"never pass  ({len(never)}): {never}")
print(f"sometimes   ({len(some)}): { {t: c[t] for t in some} }")
print("\n'sometimes' tasks are the inconsistency KiL should fix.")

## Summary

**Results** (fill in after the run)
- Complete trials: ___ / 8
- pass^1: ___ [CI ___]
- pass^8: ___ [CI ___]
- Drop from pass^1 to pass^8: ___
- Always pass / never pass / sometimes: ___ / ___ / ___
- API errors left: ___
- Time: ___ min per task at concurrency 4

**How to read it**
- pass^1 here is the mean over 8 trials, so it won't match 03's single run (0.22) exactly. A big gap means a setting changed.
- A steep drop from pass^1 to pass^8 means the agent is inconsistent. That drop is what KiL should shrink.
- "Sometimes" tasks are where consistency can improve. "Never" tasks may be too hard for any arm.

**Files**
- `results_baseline/trial_1..8/`: raw τ-bench results
- `results_baseline_passk.csv`: pass^k table
- `figures/baseline_passk.png`: the curve
- `results_baseline_config.json`: settings used (copied from `settings.py`)

**Next**
- 06: runtime validator from the policy graph, same 50 × 8 run.